# Mutual Fund Analytics - EDA Report
This notebook contains 16 comprehensive visualizations with source code.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go
import os

# Load Data
df_fund_master = pd.read_csv('data/processed/01_fund_master_cleaned.csv')
df_nav_history = pd.read_csv('data/processed/02_nav_history_cleaned.csv', parse_dates=['date'])
df_aum = pd.read_csv('data/processed/03_aum_by_fund_house_cleaned.csv', parse_dates=['date'])
df_sip_inflow = pd.read_csv('data/processed/04_monthly_sip_inflows_cleaned.csv', parse_dates=['month'])
df_cat_inflow = pd.read_csv('data/processed/05_category_inflows_cleaned.csv', parse_dates=['month'])
df_folio_count = pd.read_csv('data/processed/06_industry_folio_count_cleaned.csv', parse_dates=['month'])
df_performance = pd.read_csv('data/processed/07_scheme_performance_cleaned.csv')
df_investor = pd.read_csv('data/processed/08_investor_transactions_cleaned.csv', parse_dates=['transaction_date'])
df_holdings = pd.read_csv('data/processed/09_portfolio_holdings_cleaned.csv')

## 10 Key EDA Findings
1. **SBI Dominance**: AUM reaching over ₹12.5L Cr by 2025.
2. **SIP Surge**: Peak of ₹31,002 Cr in Dec 2025.
3. **Bull Run 2023**: Significant NAV growth across equity schemes.
4. **Market Resilience**: 2024 corrections as retail entry points.
5. **Category Preferences**: Large Cap remains top choice.
6. **Young Investors**: Strong participation from 26-45 age groups.
7. **Bharat Growth**: B30 cities contribution rising.
8. **Folio Explosion**: Nearly doubled from 2022 to 2025.
9. **Sector Focus**: Banking and IT remain leaders.
10. **Diversification**: High correlation among large-caps suggests need for alpha hunting.

### NAV Trend Analysis
Daily NAV for all schemes 2022-2026 highlighting market cycles.

In [ ]:
df_merged = df_nav_history.merge(df_fund_master[['amfi_code', 'scheme_name']], on='amfi_code')
fig = px.line(df_merged, x='date', y='nav', color='scheme_name', title='Daily NAV Trends (2022-2026)')
fig.add_vrect(x0='2023-01-01', x1='2023-12-31', fillcolor='green', opacity=0.1, annotation_text='2023 Bull Run')
fig.add_vrect(x0='2024-03-01', x1='2024-06-01', fillcolor='red', opacity=0.1, annotation_text='2024 Correction')
fig.show()

### AUM Growth
AUM growth by fund house showing SBI dominance.

In [ ]:
df_aum['year'] = df_aum['date'].dt.year
plt.figure(figsize=(12, 6))
sns.barplot(data=df_aum[df_aum['year'].isin([2022, 2023, 2024, 2025])], x='year', y='aum_crore', hue='fund_house')
plt.title('AUM Growth by Fund House (2022-2025)')
plt.annotate('SBI Dominance > ₹12.5L Cr', xy=(3, 1250000), xytext=(2, 1300000), arrowprops=dict(facecolor='black', shrink=0.05))
plt.show()

### SIP Inflow Time-series
Monthly SIP trend Jan 2022 - Dec 2025.

In [ ]:
fig = px.line(df_sip_inflow, x='month', y='sip_inflow_crore', title='Monthly SIP Inflow Trend')
fig.add_annotation(x='2025-12-01', y=31002, text='All-time High: ₹31,002 Cr', showarrow=True)
fig.show()

### Category Inflow Heatmap
Net inflow intensity by category and month.

In [ ]:
df_pivot = df_cat_inflow.pivot(index='category', columns='month', values='net_inflow_crore')
df_pivot.columns = df_pivot.columns.strftime('%Y-%m')
plt.figure(figsize=(14, 8))
sns.heatmap(df_pivot, cmap='YlGnBu')
plt.title('Net Category Inflow Heatmap')
plt.show()

### Investor Age Distribution
Distribution of investors by age group.

In [ ]:
age_dist = df_investor['age_group'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(age_dist, labels=age_dist.index, autopct='%1.1f%%', startangle=140)
plt.title('Investor Age Group Distribution')
plt.show()

### SIP Amount by Age
Box plot of SIP amounts across age groups.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(data=df_investor[df_investor['transaction_type'] == 'SIP'], x='age_group', y='amount_inr')
plt.title('SIP Amount Distribution by Age Group')
plt.show()

### Investor Gender Split
Gender distribution among investors.

In [ ]:
gender_dist = df_investor['gender'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(gender_dist, labels=gender_dist.index, autopct='%1.1f%%', startangle=140)
plt.title('Investor Gender Split')
plt.show()

### Geographic Distribution
Total SIP amount by state.

In [ ]:
state_sip = df_investor[df_investor['transaction_type'] == 'SIP'].groupby('state')['amount_inr'].sum().sort_values()
plt.figure(figsize=(10, 8))
state_sip.plot(kind='barh')
plt.title('Total SIP Amount by State')
plt.show()

### City Tier Distribution
T30 vs B30 contribution.

In [ ]:
tier_dist = df_investor['city_tier'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(tier_dist, labels=tier_dist.index, autopct='%1.1f%%', startangle=140)
plt.title('T30 vs B30 City Tier Distribution')
plt.show()

### Folio Count Growth
Industry folio growth from 13.26 Cr to 26.12 Cr.

In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(df_folio_count['month'], df_folio_count['total_folios_crore'], marker='o')
plt.title('Industry Folio Count Growth')
plt.annotate('13.26 Cr (Jan 2022)', xy=(df_folio_count['month'].iloc[0], 13.26), xytext=(df_folio_count['month'].iloc[2], 15), arrowprops=dict(facecolor='black', shrink=0.05))
plt.annotate('26.12 Cr (Dec 2025)', xy=(df_folio_count['month'].iloc[-1], 26.12), xytext=(df_folio_count['month'].iloc[-5], 24), arrowprops=dict(facecolor='black', shrink=0.05))
plt.show()

### NAV Return Correlation
Pairwise correlation of daily returns for top 10 funds.

In [ ]:
selected_codes = df_nav_history['amfi_code'].unique()[:10]
df_pivot = df_nav_history[df_nav_history['amfi_code'].isin(selected_codes)].pivot(index='date', columns='amfi_code', values='nav')
df_returns = df_pivot.pct_change().dropna()
name_map = df_fund_master.set_index('amfi_code')['scheme_name'].to_dict()
df_returns.columns = [name_map.get(c, c) for c in df_returns.columns]
plt.figure(figsize=(12, 10))
sns.heatmap(df_returns.corr(), annot=True, cmap='coolwarm', fmt='.2f')
plt.title('NAV Daily Return Correlation Matrix')
plt.show()

### Sector Allocation
Aggregate sector weights across equity funds.

In [ ]:
sector_weights = df_holdings.groupby('sector')['weight_pct'].sum().sort_values(ascending=False)
plt.figure(figsize=(10, 10))
plt.pie(sector_weights, labels=sector_weights.index, autopct='%1.1f%%', startangle=140, pctdistance=0.85)
plt.gca().add_artist(plt.Circle((0,0), 0.70, fc='white'))
plt.title('Aggregate Sector Allocation')
plt.show()

### Risk vs Return
Scatter plot of annualized risk vs return.

In [ ]:
plt.figure(figsize=(10, 8))
sns.scatterplot(data=df_performance, x='std_dev_ann_pct', y='return_3yr_pct', hue='category', size='aum_crore', sizes=(20, 500))
plt.title('Risk vs Return (3-Year Annualized)')
plt.grid(True)
plt.show()

### Expense Ratio vs AUM
Correlation between fund size and expense ratio.

In [ ]:
plt.figure(figsize=(10, 6))
sns.regplot(data=df_performance, x='aum_crore', y='expense_ratio_pct')
plt.title('Expense Ratio vs Fund Size (AUM)')
plt.show()

### Transaction Types
Distribution of transaction types (SIP vs Redemption).

In [ ]:
trans_counts = df_investor['transaction_type'].value_counts()
plt.figure(figsize=(8, 8))
plt.pie(trans_counts, labels=trans_counts.index, autopct='%1.1f%%', startangle=140)
plt.title('Transaction Type Distribution')
plt.show()

### SIP YoY Growth
Year-over-Year growth of SIP inflows.

In [ ]:
plt.figure(figsize=(12, 6))
df_sip_inflow['yoy_growth_pct'] = df_sip_inflow['sip_inflow_crore'].pct_change(periods=12) * 100
sns.lineplot(data=df_sip_inflow.dropna(), x='month', y='yoy_growth_pct', marker='o')
plt.title('Monthly SIP Inflow YoY Growth %')
plt.grid(True)
plt.show()